# Module 4: Transformer Block

### The concept

A single transformer block wraps MHA with two additions: **LayerNorm** for training stability and **residual connections** so gradients flow cleanly through depth. It also adds an **MLP sublayer** that processes each token independently after attention has mixed information across tokens.

The full block computes:

$$\text{Step 1: } X' = X + \text{MHA}(\text{LN}_1(X))$$

$$\text{Step 2: } X'' = X' + \text{MLP}(\text{LN}_2(X'))$$

where LayerNorm is defined as:

$$\text{LN}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

with $\mu$ and $\sigma$ computed **per token** (across the $D$ dimension), and $\gamma, \beta \in \mathbb{R}^D$ are learned scale and shift parameters.

The MLP sublayer applies two linear layers with a GELU activation in between:

$$\text{MLP}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

where $W_1 \in \mathbb{R}^{D \times D_{mlp}}$, $W_2 \in \mathbb{R}^{D_{mlp} \times D}$, and $D_{mlp} = 2D = 32$ in our tiny model.

GELU is approximated as:

$$\text{GELU}(x) \approx 0.5 \cdot x \cdot \left(1 + \tanh\!\left(\sqrt{\frac{2}{\pi}}\,(x + 0.044715\, x^3)\right)\right)$$

### 🤔 Pre-coding questions

**Q1.** LayerNorm computes $\mu$ and $\sigma$ per token — meaning across the $D=16$ dimension for each of the $N=16$ tokens independently. What shape are $\mu$ and $\sigma$ for our input $X \in \mathbb{R}^{B \times N \times D}$? Why does it normalize across $D$ rather than across $N$?

- Answer: The shapes of $\mu$ and $\sigma$ are then $(B,N,1)$ because the mean and std. dev are taken across the dimension $D$ for each batch and token iteration. Normalizing across $N$ would mean token $i$'s normalization depends on what *other* tokens are present in the sequence, which changes with every input. Normalizing across $D$ keeps each token's internal feature distribution stable and independent of its neighbors.

**Q2.** The residual connection adds the input directly to the output: $X' = X + \text{MHA}(\text{LN}(X))$. What specific problem does this solve during training, and what constraint does it impose on the output shape of MHA?

- Answer: Just like in ResNets, this was meant to smoothen the optimization "curves". So "graphically" speaking, we try to make the optimization easier to find the optimum by smoothing the curves. What this imposes is that the output of MHA should be the same size of the input $X$.
-  To be precise about *which* problem: in deep networks, gradients get multiplied through many weight matrices during backprop, and they tend to vanish toward zero. The residual connection creates a **direct gradient highway** — the gradient flows straight back through the addition without passing through any matrix multiplications. And yes, MHA output must be $(B, N, D)$ to match $X$ for the addition to work.

**Q3.** Notice that LN is applied before MHA and MLP (called Pre-LN), not after. The original 2017 transformer paper did it after. Why might Pre-LN be preferable, particularly in deep networks?

- Answer: **The deeper reason: with Post-LN, every** block's output gets normalized, including the residual stream itself. This means the clean gradient path through the residuals gets scaled unpredictably at each layer. With Pre-LN, the residual stream $X \rightarrow X + (\ldots)$ is always direct — normalization only happens inside the sublayer, never on the main highway. This makes gradient magnitude much more predictable across depth.

**Q4.** The MLP operates on each token independently — no information is exchanged between tokens. Given that MHA already mixes information across tokens, what is the MLP actually doing? Why is it needed at all?

- Answer: **MHA is fundamentally a linear operation** — it computes weighted sums of value vectors. No matter how many attention heads you have, you're still only doing linear combinations. The MLP is what introduces non-linearity per token, allowing the model to transform the mixed representations into complex features. Think of the division of labor as:
- MHA = where to look and what to mix across tokens
- MLP = what to do with the mixed information at each token individually
- Research actually shows that factual knowledge (e.g. "Paris is the capital of France") is largely stored in MLP weights, not attention weights.


**Q5.** $\gamma$ and $\beta$ in LayerNorm are learned parameters of shape $(D,)$. At initialization $\gamma = \mathbf{1}$ and $\beta = \mathbf{0}$, meaning LN starts as a pure normalization. What would happen to the residual stream if $\gamma = \mathbf{0}$ instead — and why does initialization matter here?

- Answer: This is to speed-up training as we further fine-tune the effects of each training step. If $\gamma = \mathbf{0}$, then:

$$\text{LN}(X) = \frac{X - \mu}{\sigma + \epsilon} \cdot \mathbf{0} + \mathbf{0} = \mathbf{0}$$

- So $\text{MHA}(\mathbf{0}) \approx \mathbf{0}$, and the residual becomes $X' = X + \mathbf{0} = X$. Every block becomes an identity function — the MHA and MLP weights receive zero gradient and learn nothing at all. With $\gamma = \mathbf{1}$, normalization is active from step one and every parameter in the block receives a meaningful gradient signal immediately.

# Coding Exercise

In [2]:
import numpy as np

# ── Tiny model constants (same as before) ─────────────────────
B, N, D = 2, 16, 16
h       = 2          # attention heads
d_k     = D // h     # 8 per head
D_mlp   = 2 * D      # 32

# ── LayerNorm parameters ──────────────────────────────────────
gamma = np.ones(D)     # scale  (D,)
beta  = np.zeros(D)    # shift  (D,)

# ── MLP parameters ────────────────────────────────────────────
W1 = np.random.randn(D, D_mlp) * 0.02   # (16, 32)
b1 = np.zeros(D_mlp)                     # (32,)
W2 = np.random.randn(D_mlp, D) * 0.02   # (32, 16)
b2 = np.zeros(D)                         # (16,)

# ── Weight matrices (one set per head) ────────────────────────
np.random.seed(42)
WQ = np.random.randn(h, D, d_k) * 0.02   # (2, 16, 8)
WK = np.random.randn(h, D, d_k) * 0.02
WV = np.random.randn(h, D, d_k) * 0.02
WO = np.random.randn(h * d_k, D) * 0.02  # output projection (16, 16)

# ── Random input head (ideally from the top) ──────────────────
X = np.random.randn(B, N, D) * 0.02

# ─────────────────────────────────────────────────────────────
# From module 3
# ─────────────────────────────────────────────────────────────

def softmax(x, axis=-1):
    """
    Numerically stable softmax along a given axis.
    Hint: subtract the max before exponentiating.
    """
    x_max = np.max(x, axis=axis, keepdims=True)   # subtract max for stability
    e_x   = np.exp(x - x_max)                     # numerator
    return e_x / np.sum(e_x, axis=axis, keepdims=True)  # normalize

def single_head_attention(X, Wq, Wk, Wv):
    """
    Single head scaled dot-product attention.

    Input:  X   shape (B, N, D)
            Wq  shape (D, d_k)
            Wk  shape (D, d_k)
            Wv  shape (D, d_k)
    Output: shape (B, N, d_k)

    Steps:
        1. Project X into Q, K, V
        2. Compute raw attention scores QK^T
        3. Scale by 1/sqrt(d_k)
        4. Softmax over the key dimension (axis=-1)
        5. Weighted sum over V
    """
    Q = X @ Wq                              # (B, N, d_k)
    K = X @ Wk                              # (B, N, d_k)
    V = X @ Wv                              # (B, N, d_k)
    A = Q @ K.transpose(0, 2, 1)            # (B, N, N)
    A_hat = softmax(A / np.sqrt(d_k))       # (B, N, N)
    out = A_hat @ V                         # (B, N, d_k)
    return out

def multi_head_attention(X, WQ, WK, WV, WO):
    """
    Multi-head attention.

    Input:  X    shape (B, N, D)
            WQ   shape (h, D, d_k)
            WK   shape (h, D, d_k)
            WV   shape (h, D, d_k)
            WO   shape (h*d_k, D)
    Output: shape (B, N, D)

    Steps:
        1. Run single_head_attention for each head
        2. Concatenate head outputs along last axis → (B, N, h*d_k)
        3. Project through WO → (B, N, D)
    """
    head_outputs = []
    for i in range(h):
        head_out = single_head_attention(X, WQ[i], WK[i], WV[i])  # (B, N, d_k)
        head_outputs.append(head_out)
    concat_heads = np.concatenate(head_outputs, axis=-1)  # (B, N, h*d_k)
    out = concat_heads @ WO                               # (B, N, D)
    return out

# ─────────────────────────────────────────────────────────────
# Coding proper: LayerNorm, GELU, MLP, and full transformer block
# ─────────────────────────────────────────────────────────────

def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Input:  x     shape (B, N, D)
            gamma shape (D,)
            beta  shape (D,)
    Output: shape (B, N, D)

    Steps:
        1. Compute mean across D axis → shape (B, N, 1)
        2. Compute std  across D axis → shape (B, N, 1)
        3. Normalize x
        4. Scale and shift with gamma, beta
    """
    mean = x.mean(axis=-1, keepdims=True)  # (B, N, 1)
    std  = x.std(axis=-1, keepdims=True)   # (B, N, 1)
    x_hat = (x - mean) / (std + eps)       # (B, N, D)
    out = gamma * x_hat + beta              # (B, N, D)
    return out


def gelu(x):
    """
    GELU activation — apply elementwise.
    Formula given in the module description above.
    """
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


def mlp(x, W1, b1, W2, b2):
    """
    Input:  x  shape (B, N, D)
    Output:    shape (B, N, D)

    Steps:
        1. Linear: x @ W1 + b1  → (B, N, D_mlp)
        2. GELU
        3. Linear: x @ W2 + b2  → (B, N, D)
    """
    l1 = x @ W1 + b1    # (B, N, D_mlp)
    a1 = gelu(l1)       # (B, N, D_mlp)
    l2 = a1 @ W2 + b2   # (B, N, D)
    return l2


def transformer_block(X, WQ, WK, WV, WO,
                       W1, b1, W2, b2,
                       gamma1, beta1,
                       gamma2, beta2):
    """
    Full Pre-LN transformer block.

    Input:  X shape (B, N, D)
    Output:   shape (B, N, D)

    Steps:
        1. X' = X  + MHA(LN1(X))
        2. X''= X' + MLP(LN2(X'))
    """
    Xp = X + multi_head_attention(layer_norm(X, gamma1, beta1), WQ, WK, WV, WO)
    Xpp = Xp + mlp(layer_norm(Xp, gamma2, beta2), W1, b1, W2, b2)
    return Xpp


# ── Shape check ───────────────────────────────────────────────
gamma1, beta1 = np.ones(D), np.zeros(D)
gamma2, beta2 = np.ones(D), np.zeros(D)

out_block = transformer_block(X, WQ, WK, WV, WO,
                               W1, b1, W2, b2,
                               gamma1, beta1,
                               gamma2, beta2)

assert out_block.shape == (B, N, D), f"Got {out_block.shape}"
print("block out:", out_block.shape)   # expect (2, 16, 16)

# ── Sanity check: LN mean and std ─────────────────────────────
ln_out = layer_norm(X, gamma1, beta1)
means = ln_out.mean(axis=-1)
stds  = ln_out.std(axis=-1)
print("LN means (should be ≈0):", means.round(4))
print("LN stds  (should be ≈1):", stds.round(4))

block out: (2, 16, 16)
LN means (should be ≈0): [[-0.  0.  0.  0.  0. -0. -0. -0. -0. -0.  0. -0.  0.  0. -0.  0.]
 [ 0. -0.  0.  0. -0.  0. -0. -0.  0. -0.  0.  0.  0.  0. -0.  0.]]
LN stds  (should be ≈1): [[0.9995 0.9994 0.9995 0.9995 0.9996 0.9995 0.9994 0.9995 0.9997 0.9994
  0.9994 0.9994 0.9995 0.9996 0.9995 0.9995]
 [0.9996 0.9992 0.9994 0.9993 0.9996 0.9995 0.9995 0.9995 0.9994 0.9995
  0.9995 0.9994 0.9994 0.9996 0.9995 0.9995]]


# Special Notes

## Where did the GeLU come from?

- GELU (Gaussian Error Linear Unit) was introduced in 2016 by Hendrycks & Gimpel. The motivation was to combine the non-linearity of ReLU with stochastic regularization ideas from dropout — in a single deterministic function.
- The idea: instead of hard-gating inputs at zero like ReLU does, weight each input by the probability that it is greater than a random Gaussian draw:

$$\text{GELU}(x) = x \cdot P(X \leq x) = x \cdot \Phi(x)$$

where $\Phi(x)$ is the Gaussian CDF — which has no closed form, hence the tanh approximation you see in the formula.

## What's wrong with ReLU in transformers
ReLU is:

$$\text{ReLU}(x) = \max(0, x)$$

It has two properties that cause friction in transformers:
1. Hard zero at the gate. ReLU kills any negative activation completely — gradient is exactly zero for $x < 0$. In deep networks with residuals this is manageable, but in the MLP sublayer it can silence large fractions of neurons, making training brittle.
2. Non-smooth at $x = 0$. The kink at zero means the gradient is discontinuous there. Optimization methods like Adam cope with this, but smoother loss landscapes generally train faster.
GELU fixes both:

$$\text{GELU}(x) \approx \begin{cases} x & x \gg 0 \\ \approx 0 & x \ll 0 \\ \text{smooth transition} & x \approx 0 \end{cases}$$

For large positive $x$, GELU ≈ $x$ like ReLU. For large negative $x$, GELU ≈ $0$ like ReLU. But around zero it transitions smoothly, and it even allows small negative values through near zero — nothing is hard-gated.

## Why transformers specifically prefer GELU
Transformers use LayerNorm heavily, which means activations entering the MLP are approximately Gaussian-distributed (mean ≈ 0, std ≈ 1) after normalization. GELU is essentially designed for exactly this distribution — its gating behavior is most meaningful when inputs are centered around zero. ReLU on a zero-centered Gaussian would kill roughly 50% of activations every forward pass.